In [1]:
import pandas as pd

# Path is relative to the notebooks/ folder, so we step up one level into data/raw/
file_path = "../data/raw/P2P_Macro_Data.csv"

# Peek at just the first 1000 rows so nothing crashes
preview = pd.read_csv(file_path, nrows=1000)

print("Shape of preview:", preview.shape)
print("Number of columns:", preview.shape[1])
print("\nColumn names:")
print(preview.columns.tolist())

Shape of preview: (1000, 195)
Number of columns: 195

Column names:
['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'op

In [2]:
# Look at the candidate target variables
print("loan_status value counts:")
print(preview['loan_status'].value_counts())

print("\nbadloan value counts:")
print(preview['badloan'].value_counts())

print("\nData types overview:")
print(preview.dtypes.value_counts())

loan_status value counts:
loan_status
Fully Paid            691
Current               165
Charged Off           129
Late (31-120 days)      7
In Grace Period         5
Late (16-30 days)       2
Default                 1
Name: count, dtype: int64

badloan value counts:
badloan
0    861
1    139
Name: count, dtype: int64

Data types overview:
float64    91
int64      64
str        40
Name: count, dtype: int64


In [3]:
# Define target on resolved loans only
resolved = preview[preview['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].copy()

# 1 = bad (default/charged off), 0 = good (fully paid)
resolved['target'] = resolved['loan_status'].isin(['Charged Off', 'Default']).astype(int)

print("Rows kept (resolved only):", len(resolved), "of", len(preview))
print("\nTarget distribution:")
print(resolved['target'].value_counts())
print("\nDefault rate:", round(resolved['target'].mean() * 100, 1), "%")

Rows kept (resolved only): 821 of 1000

Target distribution:
target
0    691
1    130
Name: count, dtype: int64

Default rate: 15.8 %


In [4]:
# Columns that leak the outcome — only knowable AFTER origination. MUST be dropped.
leakage_cols = [
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'out_prncp', 'out_prncp_inv',
    'last_credit_pull_d', 'chargeoff_within_12_mths', 'collections_12_mths_ex_med',
]

# Pre-engineered duplicates from the source (we'll make our own) — log versions and encoded versions
engineered_cols = [c for c in preview.columns if c.startswith('log') or c.endswith('_')]

# Identifiers / free text / URLs — not useful as features
id_text_cols = ['url', 'desc', 'title', 'emp_title', 'zip_code', 'id', 'member_id', 'policy_code']

print("Leakage columns present in data:", [c for c in leakage_cols if c in preview.columns])
print("\nEngineered duplicate columns:", engineered_cols)
print("\nID/text columns present:", [c for c in id_text_cols if c in preview.columns])

total_to_drop = set([c for c in leakage_cols if c in preview.columns] + engineered_cols + [c for c in id_text_cols if c in preview.columns])
print("\nTotal columns flagged for dropping:", len(total_to_drop))
print("Columns that would remain:", preview.shape[1] - len(total_to_drop))

Leakage columns present in data: ['total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'out_prncp', 'out_prncp_inv', 'last_credit_pull_d', 'chargeoff_within_12_mths', 'collections_12_mths_ex_med']

Engineered duplicate columns: ['application_type_', 'initial_list_status_', 'purpose_', 'pymnt_plan_', 'verification_status_', 'home_ownership_', 'emp_length_', 'grade_', 'sub_grade_', 'term_', 'loan_status_', 'rankloan_status_', 'logfunded_amnt', 'logannual_inc', 'logdti', 'logtotal_acc', 'logmo_sin_old_il_acct', 'logmo_sin_rcnt_tl', 'lognum_accts_ever_120_pd', 'logpct_tl_nvr_dlq', 'logpercent_bc_gt_75', 'logtot_hi_cred_lim', 'logearnings', 'logemployment', 'loglabor_force', 'logfinanceuser', 'logsocialnetworkuser', 'loginternetuser', 'logempl_expan', 'logempl_birth', 'lognew_bus', 'logpopestimate', 'logunemployment', 'loggdppercap', 'logCPIUS', 'logloan_vol', '

In [5]:
# Curated KEEP list — genuine loan & borrower attributes known at origination
loan_features = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'addr_state', 'dti', 'application_type',
]

credit_history_features = [
    'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies',
    'tot_cur_bal', 'avg_cur_bal', 'tot_hi_cred_lim', 'num_actv_bc_tl',
    'num_actv_rev_tl', 'acc_open_past_24mths', 'mo_sin_old_rev_tl_op',
    'pct_tl_nvr_dlq', 'num_tl_90g_dpd_24m', 'tax_liens',
]

# A small, defensible set of macro features for the extension
macro_features = ['FEDFUNDS', 'unem_rate', 'gdppercap', 'inf']

# The columns we need to BUILD the target (we'll drop loan_status after creating target)
target_source = ['loan_status']

keep_cols = loan_features + credit_history_features + macro_features + target_source

# Check they all exist in the data
missing = [c for c in keep_cols if c not in preview.columns]
print("Requested columns NOT found (fix these):", missing)
print("\nTotal features to keep (excl. target source):", len(keep_cols) - 1)
print("Loan features:", len(loan_features))
print("Credit history features:", len(credit_history_features))
print("Macro features:", len(macro_features))

Requested columns NOT found (fix these): []

Total features to keep (excl. target source): 38
Loan features: 14
Credit history features: 20
Macro features: 4


In [6]:
import pandas as pd

file_path = "../data/raw/P2P_Macro_Data.csv"

keep_cols = loan_features + credit_history_features + macro_features + ['loan_status']

print("Loading full dataset (39 columns, ~2.7M rows)... this may take a minute.")
df = pd.read_csv(file_path, usecols=keep_cols)
print("Loaded shape:", df.shape)

# Apply target definition: resolved loans only
before = len(df)
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].copy()
df['target'] = df['loan_status'].isin(['Charged Off', 'Default']).astype(int)
df = df.drop(columns=['loan_status'])  # drop source once target is built

print(f"\nRows before filter: {before:,}")
print(f"Rows after resolved-only filter: {len(df):,}")
print(f"Dropped (unresolved/in-progress): {before - len(df):,}")
print(f"\nDefault rate: {df['target'].mean()*100:.1f}%")
print(f"Target counts:\n{df['target'].value_counts()}")

Loading full dataset (39 columns, ~2.7M rows)... this may take a minute.
Loaded shape: (2703430, 39)

Rows before filter: 2,703,430
Rows after resolved-only filter: 943,345
Dropped (unresolved/in-progress): 1,760,085

Default rate: 18.9%
Target counts:
target
0    765339
1    178006
Name: count, dtype: int64
